In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

In [2]:
df=pd.read_csv(r'C:\Users\ADMIN\OneDrive\Desktop\5thsem_OJT\project1_diabetes\data\processed\dilna_P1W1_features.csv')

In [3]:
feature_cols=['age','pregnancies','glucose','blood_pressure','skin_thickness',
'insulin','bmi','diabetes_pedigree','bmi category','glucose category']

x=df[feature_cols].copy()
y=df['outcome']
x

,age,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,bmi category,glucose category
0,79,0,124.0,71.0,30.0,124.0,23.4,0.588,normal,prediabetic
1,37,0,153.0,85.0,24.0,42.0,33.9,0.192,obese,diabetic
2,39,0,142.0,68.0,22.0,159.0,26.9,0.777,over weight,diabetic
3,68,7,121.0,88.0,26.0,124.0,29.0,1.217,over weight,prediabetic
4,75,0,107.0,84.0,31.0,101.0,21.2,0.278,normal,prediabetic
...,...,...,...,...,...,...,...,...,...,...
945,64,2,117.0,81.0,34.0,124.0,40.3,0.641,obese,prediabetic
946,71,0,140.0,64.0,37.0,124.0,33.4,0.116,obese,diabetic
947,40,0,130.0,73.0,26.0,258.0,20.1,0.709,normal,diabetic
948,52,4,123.0,76.0,22.0,189.0,38.9,0.540,obese,prediabetic


In [4]:
x=pd.get_dummies(x,drop_first=True)
x.shape

(950, 13)

In [5]:
x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)

In [6]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [7]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dtree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [8]:
def score_row(name, pred, scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        'model' : name,
        'accuracy' : round(accuracy_score(y_test, pred),4),
        'precision' : round(precision_score(y_test, pred, zero_division=0),4),
        'recall' : round(recall_score(y_test, pred, zero_division=0),4),
        'f1' : round(f1_score(y_test,pred,zero_division=0),4),
        'found' : int(tp), 'missed' : int(fn), 'false_alarms' : int(fp)
    }

    if scores is not None :
        r['auc'] = roc_auc_score(y_test, scores)
    return r


print('train:', x_train.shape, 'test:', x_test.shape)
print('Diabetic patients in the test set:', int(np.sum(y_test)),'of', len(y_test))
for nm, m, Xt in [('Logistic Regression', logreg, x_test_scaled),('Decision Tree (d=3)', dtree, x_test),('KNN (k=25)', knn, x_test_scaled)]:
    print(f' {nm:22s} accuracy {accuracy_score(y_test, m.predict(Xt)):.4f}')

train: (760, 13) test: (190, 13)
Diabetic patients in the test set: 55 of 190
 Logistic Regression    accuracy 0.7526
 Decision Tree (d=3)    accuracy 0.7211
 KNN (k=25)             accuracy 0.7474


RESTATE THE TARGET YOU ARE AIMING

In [9]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182  # Week 5 Logistic Regression at threshold 0.30

print("PROJECT METRIC : recall  (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall  (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


How an SVM Makes a Decision
An SVM (Support Vector Machine) classifies data by finding a boundary between groups.
Hyperplane: The boundary that separates the groups. In a 2D graph, it is a line; in higher dimensions, it is called a hyperplane.
Support vectors: The data points closest to the hyperplane. They help determine where the boundary is placed.
Margin: The distance between the hyperplane and the nearest support vectors. The SVM tries to make this margin as wide as possible.
Decision process
The SVM finds a hyperplane that separates the groups.
It uses the support vectors to position the hyperplane.
It chooses a hyperplane with a wide margin.
For a new data point, it checks which side of the hyperplane the point falls on and predicts that group.

## train an SVM with default settings

In [10]:
from sklearn.svm import SVC

svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled, y_train)

pred_rbf = svm_rbf.predict(x_test_scaled)

print("Fitted:", svm_rbf)
print("Patients flagged:", int(pred_rbf.sum()), "of", len(pred_rbf))

Fitted: SVC(random_state=42)
Patients flagged: 17 of 190


In [11]:
row_rbf = score_row("SVC rbc (default)", pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("this model  :", row_rbf["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarms     auc
SVC rbc (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.64431

target recall: 0.7 | best so far: 0.6182
this model  : 0.1818


In [12]:
svm_linear = SVC(kernel="linear",random_state=42)
svm_linear.fit(x_train_scaled,y_train)
pred_svm = svm_linear.predict(x_test_scaled)
row_lin = score_row("SVC rbc (default) ",pred_rbf,
                    svm_linear.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf,row_lin]).to_string(index=False))


             model  accuracy  precision  recall     f1  found  missed  false_alarms      auc
 SVC rbc (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.644310
SVC rbc (default)     0.7263     0.5882  0.1818 0.2778     10      45             7 0.692795


In [13]:
print("Patients the linear SVM flagged as diabetic:", int(pred_svm.sum()), "of", len(pred_svm))
print()
print("Linear SVM confusion matrix:")
print(confusion_matrix(y_test, pred_svm))
print()
print("Week 5 dummy model confusion matrix:")
print(confusion_matrix(y_test, np.zeros_like(y_test)))
print()
print("Linear SVM accuracy :", round(accuracy_score(y_test, pred_svm), 4))
print("Dummy model accuracy:", round(accuracy_score(y_test, np.zeros_like(y_test)), 4))

Patients the linear SVM flagged as diabetic: 0 of 190

Linear SVM confusion matrix:
[[135   0]
 [ 55   0]]

Week 5 dummy model confusion matrix:
[[135   0]
 [ 55   0]]

Linear SVM accuracy : 0.7105
Dummy model accuracy: 0.7105


## class weight

learn what class weight does

class_weight gives different importance to different classes during model training.

It is mainly used for imbalanced datasets, giving more weight to the minority class.

class_weight="balanced" automatically gives higher importance to the minority class.

In [14]:
svm_rbf_bal = SVC(random_state=42, class_weight="balanced").fit(
    x_train_scaled, y_train)
svm_lin_bal = SVC(kernel="linear", random_state=42,
                  class_weight="balanced").fit(
    x_train_scaled, y_train)
rows = [
    row_rbf,
    score_row(
        "SVC rbf (balanced)",
        svm_rbf_bal.predict(x_test_scaled),
        svm_rbf_bal.decision_function(x_test_scaled)
    ),
    row_lin,
    score_row(
        "SVC linear (balanced)",
        svm_lin_bal.predict(x_test_scaled),
        svm_lin_bal.decision_function(x_test_scaled)
    )]
svm_table = pd.DataFrame(rows)
print(svm_table.to_string(index=False))

                model  accuracy  precision  recall     f1  found  missed  false_alarms      auc
    SVC rbc (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.644310
   SVC rbf (balanced)    0.6368     0.4146  0.6182 0.4964     34      21            48 0.682559
   SVC rbc (default)     0.7263     0.5882  0.1818 0.2778     10      45             7 0.692795
SVC linear (balanced)    0.6579     0.4375  0.6364 0.5185     35      20            45 0.711380


## get the risk score out of the svm

In [17]:
score_bal = svm_rbf_bal.decision_function(x_test_scaled)
print()
print("decision_function output for the first 8 test patients")
print(np.round(score_bal[:8],4))
print()
print("negative = healthy side of the boundary , positive = diabetic side")
print('AUC from decision_function:',
       round(roc_auc_score(y_test,score_bal),4))
print()
print("AUC of the linear SVM that nobody:",
      round(roc_auc_score(y_test,svm_linear.decision_function(x_test_scaled)),4))


decision_function output for the first 8 test patients
[ 0.0272 -1.5739 -0.7555 -0.5524 -0.74   -1.247   0.7715  1.0989]

negative = healthy side of the boundary , positive = diabetic side
AUC from decision_function: 0.6826

AUC of the linear SVM that nobody: 0.6928


 check how much of the data the boundary rests on

In [20]:
print("support vectors", svm_rbf.n_support_)
print("total support",svm_rbf.n_support_.sum(),'of',len(y_train),
      f"={svm_rbf.n_support_.sum() / len(y_train) * 100:.1f}% of the training set")

support vectors [277 212]
total support 489 of 760 =64.3% of the training set


try one change to c

In [24]:
svm_c10 = SVC(random_state=42, C=10).fit(x_train_scaled, y_train)

print(pd.DataFrame([
    row_rbf,
    score_row(
        "SVC rbf C=10",
        svm_c10.predict(x_test_scaled),
        svm_c10.decision_function(x_test_scaled)
    )
]).to_string(index=False))

            model  accuracy  precision  recall     f1  found  missed  false_alarms      auc
SVC rbc (default)    0.7263     0.5882  0.1818 0.2778     10      45             7 0.644310
     SVC rbf C=10    0.6684     0.3947  0.2727 0.3226     15      40            23 0.621279
